In [ ]:
# Install Tesseract OCR engine and Python wrapper
!sudo apt-get install tesseract-ocr
!pip install pytesseract
!pip install opencv-python
!pip install pdf2image
!apt-get install poppler-utils  # Required if you are processing the actual PDF file

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 1s (195 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processin

In [ ]:
import cv2
import pytesseract
from pytesseract import Output
import numpy as np
import re
import json
from PIL import Image
from pdf2image import convert_from_path
import os

# --- CLASS DEFINITION ---
class MortgageParser:
    def __init__(self, image_path):
        self.image_path = image_path
        # Read image using OpenCV
        self.image = cv2.imread(image_path)
        if self.image is None:
            raise FileNotFoundError(f"OpenCV could not read the image: {image_path}")

        self.gray = None
        self.thresh = None
        self.ocr_data = None
        self.full_text = ""

    def preprocess_image(self):
        # Convert to grayscale
        self.gray = cv2.cvtColor(self.image, cv2.COLOR_BGR2GRAY)
        # Apply thresholding
        _, self.thresh = cv2.threshold(self.gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
        # Denoising
        self.thresh = cv2.fastNlMeansDenoising(self.thresh, None, 10, 7, 21)
        return self.thresh

    def run_ocr(self):
        # Get detailed data including bounding boxes
        self.ocr_data = pytesseract.image_to_data(self.thresh, output_type=Output.DICT)
        # Get simple string for Regex parsing
        self.full_text = " ".join([text for text in self.ocr_data['text'] if text.strip() != ''])
        return self.full_text

    def find_bounding_box_for_phrase(self, phrase):
        if not phrase: return None
        words = phrase.split()
        if not words: return None

        n_boxes = len(self.ocr_data['text'])
        for i in range(n_boxes):
          curr_text = self.ocr_data['text'][i].strip()
          if curr_text and (curr_text in words[0] or words[0] in curr_text):
                return [
                    self.ocr_data['left'][i],
                    self.ocr_data['top'][i],
                    self.ocr_data['width'][i],
                    self.ocr_data['height'][i]
                ]
        return None

    def extract_information(self):
        extracted_data = {}
        patterns = {
            "document_title": r"(MORTGAGE|DEED OF TRUST)",
            "loan_amount": r"U\.S\.\s?\$([0-9,]+\.[0-9]{2})",
            "borrower_name": r"Mortgagor is\s+([A-Z\s]+),",
            "lender_name": r"Lender is\s+([A-Za-z\s&]+?)\s+and",
            "loan_date": r"given on\s+([A-Za-z]+\s\d{1,2},\s\d{4})"
        }

        for key, pattern in patterns.items():
            match = re.search(pattern, self.full_text, re.IGNORECASE)
            if match:
                value = match.group(1).strip()
                bbox = self.find_bounding_box_for_phrase(value)
                extracted_data[key] = {"value": value, "bounding_box": bbox}
            else:
                extracted_data[key] = None
        return extracted_data

# --- EXECUTION ---
# This variable was missing in your previous run:
input_filename = "MTG_10009588.pdf"

try:
    print(f"Starting process for: {input_filename}")
    image_path = input_filename

    # Check if file exists
    if not os.path.exists(input_filename):
        raise FileNotFoundError(f"The file '{input_filename}' was not found in Colab files.")

    # Convert PDF to Image if necessary
    if input_filename.lower().endswith('.pdf'):
        print("Converting PDF to Image...")
        pages = convert_from_path(input_filename)
        if pages:
            image_path = "temp_converted.jpg"
            pages[0].save(image_path, 'JPEG')
            print("Conversion successful.")
        else:
            raise Exception("PDF conversion failed. File might be empty.")

    # Run the Parser
    parser = MortgageParser(image_path)
    parser.preprocess_image()
    text = parser.run_ocr()

    # Print preview
    print(f"\n--- OCR Extract (First 200 chars) ---\n{text[:200]}...\n")

    # Extract JSON
    final_json = parser.extract_information()
    print("--- Structured JSON Output ---")
    print(json.dumps(final_json, indent=4))

except Exception as e:
    print(f"\nCRITICAL ERROR: {e}")
    print("Troubleshooting: Ensure 'MTG_10009588.pdf' is uploaded in the Files tab on the left.")

Starting process for: MTG_10009588.pdf
Converting PDF to Image...
Conversion successful.

--- OCR Extract (First 200 chars) ---
- WERRAYE MORTGAGE DOC.# 10009588 DOCUMENT NUMBER, RECORDED 06/28/2011 09:35AM JOHN LA FAVE NAME & RETURN ADDRESS REGISTER DF DEEDS M&I Home Lending Solutions Milwaukee County, WII Attn: Secondary Mar...

--- Structured JSON Output ---
{
    "document_title": {
        "value": "MORTGAGE",
        "bounding_box": [
            746,
            130,
            213,
            55
        ]
    },
    "loan_amount": null,
    "borrower_name": {
        "value": "KIMBERLY HOGAN",
        "bounding_box": [
            508,
            755,
            97,
            17
        ]
    },
    "lender_name": null,
    "loan_date": {
        "value": "June 20, 2011",
        "bounding_box": [
            891,
            730,
            47,
            16
        ]
    }
}
